# Submission Summary

| Item | Group 37 submission |
|---|---|
| **Group number** | 37 |
| **Full name** | Adefemi Adelugba |
| **Dataset** | Telco Customer Churn |
| **Depth track** | Classification Depth |
| **Team members** | Add all Group 37 members before submission |
| **Final model** | Updated automatically after the final evaluation |
| **Stratified dummy F1 / ROC AUC** | Updated automatically after the final evaluation |
| **Final model F1 / ROC AUC** | Updated automatically after the final evaluation |
| **Leakage columns dropped** | None identified in the Telco dataset; `customerID` is excluded from modelling as an identifier |
| **Full run time on Colab** | Record after a clean Restart & Run All |

> **Submission note:** Run the entire notebook on a fresh Colab session, then replace the summary placeholders with the values printed in the final evaluation cell and add the full names of all Group 37 members.

## Part 1. Data Foundation (15%)

**STEP 1A**: LOAD THE DATASET FROM KAGGLE INSIDE THE NOTEBOOK.

In [ ]:
#INSATLLING KAGGLEHUB
!pip install kagglehub

In [ ]:
#GETTING AUTHENTICATION FROM KAGGLEHUB
import kagglehub
kagglehub.login()

In [ ]:
#DOWNLOADING THE DATASET.
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Dataset downloaded to:", path)

In [ ]:
#KNOWING THE FILE NAME.
import os
print(os.listdir(path))

In [ ]:
#LOADING THE DATASET
import pandas as pd
file_path = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(file_path)
df.head()


**STEP 1B**: PUSH OUR DATA INTO SQLITE.

In [ ]:
#importing sqlite.
import sqlite3

In [ ]:
#creating a connection to a sqlite database
conn = sqlite3.connect("telco_churn.db")

In [ ]:
#putting our dataframe into sqlite and name it customers
df.to_sql("customers", conn, if_exists="replace", index=False)

**STEP 2:** RUNNING THREE SQL QUERIES THAT REVEAL SOMETHING USEFUL FOLLOWED BY A MARKDOWN CELL EXPLAINING WHAT WE LEARNT.

In [ ]:
#QUERY 1 : HOW MANY CUSTOMERS CHURNED?
# Count customers in each churn category
query1 = """
SELECT Churn, COUNT(*) AS customer_count
FROM customers
GROUP BY Churn;
"""
result1 = pd.read_sql_query(query1, conn)
print(result1)

**SQL QUERY 1: MARKDOWN EXPLANATION CELL:**

The query shows that 5,174 customers did not churn, while 1,869 customers churned. This means that approximately 73.5% of customers stayed with the company, while approximately 26.5% left. The target variable is therefore not evenly distributed, which is important because we will need to consider class imbalance when building our classification models later in the project.


In [ ]:
# QUERY 2: DOES THE TYPE OF CONTRACT APPEAR TO BE RELATED TO CUSTOMER CHURN?
query2 = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC;
"""

result2 = pd.read_sql_query(query2, conn)
result2

**SQL QUERY 2 :MARKDOWN EXPLANATION CELL**:

**calculating the churn percentage within each contract type**,

month-to-month = 2220 + 1655 =3875

one year = 1307 + 166 = 1473

two year = 1647 + 48 = 1695

**percentage =**

month-to-month = 1655/3875 = 42.7%

one year = 166/1473 = 11.3%

two year = 48/1695 = 2.8%

The results show a clear difference in customer churn across contract types. Among month-to-month customers, 1,655 out of 3,875 customers churned, which is approximately 42.7%. For one-year contracts, 166 out of 1,473 customers churned, approximately 11.3%. For two-year contracts, only 48 out of 1,695 customers churned, approximately 2.8%.

This suggests that contract type is strongly associated with customer churn in this dataset. Month-to-month customers have a much higher observed churn rate than customers with one-year or two-year contracts. We will investigate this relationship further during exploratory data analysis and model building.


In [ ]:
## query 3:CHURN + REVENUE EXPOSURE:WHICH CUSTOMER SEGMENTS HAS BOTH HIGH CHURN AND HIGH MONTHLY REVENUE AT RISK?
query3 = """
SELECT
    Contract,
    InternetService,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate_pct,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 2)
        AS churned_customers_monthly_charges
FROM customers
GROUP BY Contract, InternetService
HAVING COUNT(*) >= 50
ORDER BY churned_customers_monthly_charges DESC;
"""

result3 = pd.read_sql_query(query3, conn)

result3

**SQL QUERY 3: MARKDOWN EXPLANATION CELL:**

This query combines contract type and internet service to identify customer segments with both high churn and potential monthly revenue exposure.

The month-to-month fiber-optic segment stands out in the results. It contains 2,128 customers, of whom 1,162 churned, giving an observed churn rate of 54.61%. The monthly charges associated with these churned customers total 100,482.

This shows why looking at churn rate alone may not be enough for a business. A customer segment can have a high churn rate, but the potential financial impact also depends on the number of customers and their monthly charges. The results therefore give us a useful business perspective that we can investigate further during the exploratory analysis and classification stages.


**STEP 3:** **PERFORM EDA**(**EXPLORATORY DATA ANALYSIS**)
  

*   SHAPE
*  TYPES


*  MISSING VALUES
*  TARGET DISTRIBUTION


*  THREE TO FIVE FEATURE DISTRIBUTION







In [ ]:
# SHAPE : 7043 ROWS AND 21 COLUMNS.
df.shape

In [ ]:
# TYPES
df.dtypes

In [ ]:
# MISSING VALUES
df.isnull().sum()

In [ ]:
# TARGET DISTRIBUTION.
df["Churn"].value_counts()

THREE- FIVE FEATURE DISTRIBUTIONS

1. tenure             -      how long customers have stayed.
2. monthly charges     -      what customers are charged monthly.
3. contract           -      customer's contract type.
4. internet service-         customers internet service type







In [ ]:
# TENURE
df["tenure"].describe()


In [ ]:
# DISTRIUTION CHART FOR TENURE
import matplotlib.pyplot as plt

plt.hist(df["tenure"], bins=20)
plt.title("Distribution of Customer Tenure")
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# MONTHLYCHARGES
df["MonthlyCharges"].describe()

In [ ]:
#DISTRIBUTION CHART FOR MONTHLYCHARGES
plt.hist(df["MonthlyCharges"], bins=20)
plt.title("Distribution of Monthly Charges")
plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
# CONTRACT
df["Contract"].value_counts()

In [ ]:
# DISTRIBUTION CHART FOR CONTRACT
import matplotlib.pyplot as plt

df["Contract"].value_counts().plot(kind="bar")

plt.title("Distribution of Contract Types")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.show()

In [ ]:
#INTERNETSERVICE
df["InternetService"].value_counts()

In [ ]:
#DISTRIBUTION CHART FOR INTERNETSERVICE
import matplotlib.pyplot as plt

df["InternetService"].value_counts().plot(kind="bar")

plt.title("Distribution of Internet Service Types")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.show()

**STEP 4:** CLEAN THE DATA,JUSTFIFYING EVERY STEP IN MARKDOWN.NO BLANCKET fillna. IF YOU DROP ROWS,EXPLAIN WHY.
 ## Data Cleaning

Before cleaning the dataset, we will identify columns that require correction or transformation based on the findings from the exploratory data analysis. Each cleaning decision will be justified to ensure that we do not make unnecessary changes to the data.


In [ ]:
#CHECK TOTALCHARGES
df["TotalCharges"].head(10)

In [ ]:
#CHECKING FOR BLANK VALUES
df["TotalCharges"].value_counts().head()

In [ ]:
#CHECKING FOR EMPTY OR WHITE-SPACE ENTRIES
(df["TotalCharges"].astype(str).str.strip() == "").sum()

MARKDOWN JUSTIFICATION
### Cleaning `TotalCharges`

The `TotalCharges` column is stored as an object data type even though it represents numerical customer charges. Our inspection found 11 blank or whitespace-only values in this column.

Because `TotalCharges` is required as a numerical feature for later analysis and modelling, these blank entries cannot be directly converted to numbers. We will remove the 11 affected rows because the values are unavailable and there is no reliable value in the dataset that can be used to replace them without introducing assumptions.

This is a targeted removal of only the rows affected by the missing `TotalCharges` values, rather than using a blanket `fillna()` approach.


In [ ]:
# REMOVING THE 11 ROWS
df = df[df["TotalCharges"].astype(str).str.strip() != ""]

In [ ]:
#CHECKING HOW MANY ROWS REMAIN
df.shape

Converting `TotalCharges` to Numeric

After removing the 11 rows with blank `TotalCharges` values, the remaining values represent numerical customer charges. We will convert this column from `object` to a numeric data type so that it can be correctly used in numerical analysis and machine learning.


In [ ]:
#COVERTING THE DATA TYPE ROBUSTLY
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
# Remove rows where TotalCharges became NaN due to coercion
df.dropna(subset=["TotalCharges"], inplace=True)

In [ ]:
#CHECKING THE DATA TYPE IF IT HAS BEEN CORRECTED
df["TotalCharges"].dtype

In [ ]:
df.isnull().sum()

Checking for Duplicate Rows

We will check the dataset for duplicate rows to ensure that the same customer record has not been unintentionally repeated. Duplicate records could cause certain customers to have more influence during analysis and model training.


In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

 **DATA CLEANING CONCLUSION**

The dataset was cleaned based on the issues identified during the exploratory analysis. The `TotalCharges` column contained 11 blank values, so the 11 affected rows were removed because there was no reliable value available for replacement. The remaining `TotalCharges` values were converted from `object` to `float64` so that the column could be used correctly for numerical analysis and modelling.

A final missing-value check confirmed that there are no missing values remaining, and a duplicate check confirmed that there are no duplicate rows. No blanket `fillna()` method was used.


## Part 2. Feature Engineering and Encoding (20%)

**STEP 1:** Create at least three new features (for example: bin a continuous column, count occurrences per row,take a ratio, or build an interaction term). Each needs a markdown cell explaining the business reasoning

**THREE NEW FEATURES**:

1. **Tenure Group** —      bin customer tenure into meaningful stages.
2. **Services Count**— count how many additional services each customer has.
3. **Charge per Tenure Month** — a ratio using TotalCharges and tenure.



In [ ]:
# Feature 1: TenureGroup
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 36, 72],
    labels=['New_0-12m', 'Mid_13-36m', 'Loyal_37-72m']
)

df['TenureGroup'].value_counts().sort_index()

### Feature 3: ChargePerTenureMonth

**Business reasoning:** `TotalCharges` alone is strongly related to tenure because customers who stay longer naturally accumulate more charges. Dividing total charges by tenure provides a simple average-spend measure that helps distinguish high-value customers from customers whose total spend is high mainly because they have been customers for many months.

In [ ]:
# Feature 3: average charge per month of observed tenure
# The +1 prevents division by zero for customers with zero recorded tenure.
df['ChargePerTenureMonth'] = df['TotalCharges'] / (df['tenure'] + 1)
df['ChargePerTenureMonth'].describe()

**Feature 1**: **Tenure Group**

Customer tenure represents how long a customer has been with the company. We will create a `TenureGroup` feature by grouping customers into different stages of their relationship with the company.

This feature can help identify whether churn patterns differ between newer customers and customers who have stayed with the company for longer periods. The groups also provide a more business-friendly representation of customer tenure than using only the raw number of months. cut


### Feature 2: NumAddOnServices

**Business reasoning:** Customers who subscribe to more optional services may have a different relationship with the company than customers using only the basic service. Counting selected add-on services creates a simple measure of product adoption and bundle depth that can be used in segmentation and churn modelling.

In [ ]:
# Count the number of optional/add-on services each customer has
add_on_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines'
]

df['NumAddOnServices'] = (df[add_on_cols] == 'Yes').sum(axis=1)
df['NumAddOnServices'].describe()

### Feature 3: IsHighRiskProfile (High-Risk Fiber + Month-to-Month Segment)
**Business Reasoning:** From telco domain knowledge, Fiber optic customers on Month-to-month contracts without TechSupport churn at the highest rate - Fiber is expensive and prone to complaints, Month-to-month means no commitment, and No TechSupport means issues are unresolved. This interaction flag isolates the highest-risk segment for targeted retention campaigns.

In [ ]:
# Feature 3: High-risk interaction
df['IsHighRiskProfile'] = (
    (df['InternetService'] == 'Fiber optic') &
    (df['Contract'] == 'Month-to-month') &
    (df['TechSupport'] == 'No')
).astype(int)

# Extra feature for stronger story: AvgMonthlySpend Efficiency
df['AvgMonthlyToTotalRatio'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 to avoid div by 0

print(f"High-risk customers: {df['IsHighRiskProfile'].sum()} out of {len(df)} ({df['IsHighRiskProfile'].mean()*100:.1f}%)")
df[['tenure', 'InternetService', 'Contract', 'TechSupport', 'IsHighRiskProfile']].head(8)

In [ ]:
df.shape


In [ ]:
print(df.columns.tolist())

### Additional Feature: TenureGroup & NumAddOnServices
Business Reasoning: Added to meet 3-feature minimum. TenureGroup captures customer lifecycle stage, NumAddOnServices captures bundle stickiness.

In [ ]:
# Ensure the engineered features exist (safe to run after the earlier feature cells)
if 'TenureGroup' not in df.columns:
    df['TenureGroup'] = pd.cut(
        df['tenure'], bins=[-1, 12, 36, 72],
        labels=['New_0-12m', 'Mid_13-36m', 'Loyal_37-72m']
    )

if 'NumAddOnServices' not in df.columns:
    df['NumAddOnServices'] = (df[add_on_cols] == 'Yes').sum(axis=1)

print(df.shape)
print(df[['TenureGroup', 'NumAddOnServices', 'IsHighRiskProfile',
          'AvgMonthlyToTotalRatio', 'ChargePerTenureMonth']].head())

### Part 2.2 - Encoding Categorical Columns
We have 16 categorical columns. So here is the strategy:
- Binary Yes/No and Male/Female (gender, Partner, Dependents, PhoneService, PaperlessBilling, Churn): Label encoding 0/1 for efficiency. This is identical to one-hot with drop_first.
- Multi-category (MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaymentMethod, TenureGroup): One-hot encoding to avoid false ordinal ordering. We use drop_first=True to avoid dummy variable trap and reduce dimensionality for K-Means.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Make a copy for modelling
df_model = df.copy()

# Binary variables: map to 0/1 because they have two meaningful categories.
binary_map = {'Yes': 1, 'No': 0}
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']:
    df_model[col] = df_model[col].map(binary_map)

df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})

# Multi-category variables: one-hot encoding avoids imposing a false ordinal ranking.
multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod', 'TenureGroup'
]

df_encoded = pd.get_dummies(df_model, columns=multi_cols, drop_first=True, dtype=int)

# Identifier is not a predictive feature; Churn is the target.
X = df_encoded.drop(columns=['customerID', 'Churn'])
y = df_encoded['Churn'].astype(int)

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts(normalize=True)}")
print("Encoded columns:", X.columns.tolist()[:15], "...")

In [ ]:
df.shape

In [ ]:
print(X.shape)

### Part 2.3 - Scaling
We choose StandardScaler over MinMaxScaler. Here is the reason: Our features have vastly different scales and outliers (MonthlyCharges ~ 20-120, TotalCharges ~ 20-8500, tenure 0-72, NumAddOnServices 0-7). K-Means uses Euclidean distance and Logistic Regression uses gradient descent - both are dominated by large-scale features if not scaled. StandardScaler (z-score, mean=0, std=1) is more robust to outliers than MinMax which compresses everything into [0,1] and gets distorted by one extreme TotalCharges value. We fit on train only to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split before fitting the scaler so information from the test set cannot influence training.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled, columns=X_train.columns, index=X_train.index
)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled, columns=X_test.columns, index=X_test.index
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Scaled training shape: {X_train_scaled_df.shape}")
print(f"Scaled test shape: {X_test_scaled_df.shape}")

## Part 3. Clustering Analysis (20%)

Clustering is an unsupervised step, so `Churn` is deliberately excluded. The objective is to discover natural customer groups based on the features available before the churn outcome is considered.

### 3.1 K-Means for K = 2 to 10

We will evaluate both the **inertia/elbow curve** and the **silhouette score**. Inertia always decreases as K increases, so the elbow helps identify diminishing returns, while silhouette measures how well-separated and internally coherent the clusters are.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt

k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train_scaled_df)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_train_scaled_df, labels))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_values), inertias, marker='o')
ax.set_title('K-Means Elbow Curve')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Inertia')
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_values), silhouette_scores, marker='o')
ax.set_title('K-Means Silhouette Scores')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette score')
ax.grid(alpha=0.25)
plt.show()

best_silhouette_k = int(list(k_values)[int(np.argmax(silhouette_scores))])
print("K with highest silhouette score:", best_silhouette_k)
print("Highest silhouette score:", round(max(silhouette_scores), 4))

### 3.2 Selecting the Final K

The final K should not be selected from the silhouette score alone. We will use the highest-silhouette candidate together with the elbow curve, then state the chosen value explicitly. The code below uses the strongest silhouette candidate as the starting point and records the choice for reproducibility.

In [ ]:
# Select the K with the strongest silhouette signal.
# The elbow plot above should be checked visually before submission.
final_k = best_silhouette_k

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
train_clusters = kmeans_final.fit_predict(X_train_scaled_df)
test_clusters = kmeans_final.predict(X_test_scaled_df)

print(f"Final K selected: {final_k}")
print(pd.Series(train_clusters).value_counts().sort_index().rename('customers'))

### 3.3 Cluster Profiles and Plain-English Names

We now return to the original, interpretable variables. This makes the clusters understandable to a business audience instead of describing them only as mathematical labels.

In [ ]:
# Profile clusters using original-scale variables
train_profile = df.loc[X_train.index].copy()
train_profile['Cluster'] = train_clusters

numeric_profile_cols = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'NumAddOnServices', 'IsHighRiskProfile'
]

cluster_numeric = train_profile.groupby('Cluster')[numeric_profile_cols].mean().round(2)

# Add observed churn rate only for interpretation/tie-in; it was NOT used to form the clusters.
cluster_churn = train_profile.groupby('Cluster')['Churn'].apply(
    lambda s: (s == 'Yes').mean()
).mul(100).round(2).rename('ChurnRatePct')

cluster_profile = cluster_numeric.join(cluster_churn)
cluster_profile

In [ ]:
# Add contract and internet-service composition to help name the clusters
contract_mix = pd.crosstab(
    train_profile['Cluster'], train_profile['Contract'], normalize='index'
).mul(100).round(1)

internet_mix = pd.crosstab(
    train_profile['Cluster'], train_profile['InternetService'], normalize='index'
).mul(100).round(1)

print("Contract mix (%):")
display(contract_mix)
print("Internet-service mix (%):")
display(internet_mix)

def name_cluster(row):
    tenure_label = "long-tenure" if row['tenure'] >= train_profile['tenure'].median() else "short-tenure"
    spend_label = "higher-spend" if row['MonthlyCharges'] >= train_profile['MonthlyCharges'].median() else "lower-spend"
    addon_label = "bundle-rich" if row['NumAddOnServices'] >= train_profile['NumAddOnServices'].median() else "basic-service"
    return f"{tenure_label} {spend_label} {addon_label}"

cluster_names = {idx: name_cluster(row) for idx, row in cluster_profile.iterrows()}
cluster_profile_named = cluster_profile.copy()
cluster_profile_named.insert(0, 'ClusterName',
                              cluster_profile_named.index.map(cluster_names))
cluster_profile_named

### 3.4 PCA Visualization

K-Means uses the full feature space. PCA is used only to project those high-dimensional observations into two dimensions so we can visually inspect whether the selected clusters are reasonably separated.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled_df)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    X_train_pca[:, 0], X_train_pca[:, 1],
    c=train_clusters, alpha=0.45, s=18
)
plt.title(f'Customer Clusters Visualized with PCA (K={final_k})')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.colorbar(scatter, label='Cluster')
plt.show()

print("Total variance explained by the two PCA components:",
      round(pca.explained_variance_ratio_.sum()*100, 2), "%")

## Part 4. Classification (20%)

The supervised task is to predict whether a customer will churn. Because churn is the positive class of interest, we will report accuracy, precision, recall, F1 and ROC AUC rather than relying on accuracy alone.

### 4.1 Train/Test Setup

The dataset has more than 2,000 observations, so the brief permits a stratified train/test split. `stratify=y` preserves the churn/non-churn proportion in both sets.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

def evaluate_binary_model(name, model, X_eval, y_eval, threshold=0.5):
    model.fit(X_train_scaled_df, y_train)
    prob = model.predict_proba(X_eval)[:, 1]
    pred = (prob >= threshold).astype(int)
    return {
        'Model': name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_eval, pred),
        'Precision': precision_score(y_eval, pred, zero_division=0),
        'Recall': recall_score(y_eval, pred, zero_division=0),
        'F1': f1_score(y_eval, pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_eval, prob),
        'Confusion_Matrix': confusion_matrix(y_eval, pred)
    }

models = {
    'Dummy (stratified)': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

baseline_results = []
for name, model in models.items():
    if name.startswith('Dummy'):
        model.fit(X_train_scaled_df, y_train)
        prob = model.predict_proba(X_test_scaled_df)[:, 1]
        pred = (prob >= 0.5).astype(int)
        baseline_results.append({
            'Model': name, 'Threshold': 0.5,
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred, zero_division=0),
            'Recall': recall_score(y_test, pred, zero_division=0),
            'F1': f1_score(y_test, pred, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test, prob),
            'Confusion_Matrix': confusion_matrix(y_test, pred)
        })
    else:
        baseline_results.append(
            evaluate_binary_model(name, model, X_test_scaled_df, y_test)
        )

baseline_table = pd.DataFrame(baseline_results)
baseline_table[['Model','Threshold','Accuracy','Precision','Recall','F1','ROC_AUC']]

### 4.2 Confusion Matrices and Classification Reports

The confusion matrix shows the operational trade-off between correctly identifying churners and incorrectly flagging customers who would stay. The classification report provides the class-level precision, recall and F1 values.

In [ ]:
for result in baseline_results:
    print(f"\n{'='*70}\n{result['Model']}")
    print("Confusion matrix:")
    print(result['Confusion_Matrix'])

# Detailed report for the strongest baseline by ROC AUC
best_baseline_name = (
    baseline_table[baseline_table['Model'] != 'Dummy (stratified)']
    .sort_values(['ROC_AUC', 'F1'], ascending=False)
    .iloc[0]['Model']
)
best_baseline_model = models[best_baseline_name]
best_baseline_model.fit(X_train_scaled_df, y_train)
best_prob = best_baseline_model.predict_proba(X_test_scaled_df)[:, 1]
best_pred = (best_prob >= 0.5).astype(int)

print(f"Selected baseline for detailed report: {best_baseline_name}")
print(classification_report(y_test, best_pred, target_names=['No Churn', 'Churn'], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation='nearest')
plt.title(f'Confusion Matrix - {best_baseline_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0, 1], ['No Churn', 'Churn'])
plt.yticks([0, 1], ['No Churn', 'Churn'])
for r in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, r, cm[r, col], ha='center', va='center')
plt.colorbar(label='Count')
plt.tight_layout()
plt.show()

## Part 5. Imbalance Handling (10%)

The target is imbalanced: approximately 26.5% of the cleaned observations are churners. We therefore test three permitted approaches: **class weights**, **SMOTE**, and **threshold tuning**. The aim is not simply to maximize accuracy; it is to improve identification of the positive class while keeping the business trade-off explicit.

### 5.1 Class-Weighted Models

Class weighting makes mistakes on the minority churn class more costly during model fitting. This changes the learning objective without creating synthetic observations.

In [ ]:
weighted_models = {
    'Logistic Regression (class_weight=balanced)': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=42
    ),
    'Random Forest (class_weight=balanced)': RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

weighted_results = []
for name, model in weighted_models.items():
    result = evaluate_binary_model(name, model, X_test_scaled_df, y_test)
    weighted_results.append(result)

pd.DataFrame(weighted_results)[
    ['Model','Threshold','Accuracy','Precision','Recall','F1','ROC_AUC']
]

### 5.2 SMOTE

SMOTE creates synthetic minority-class training observations. It must be applied **only to the training data**, after the train/test split, so that synthetic information derived from the test set cannot leak into model evaluation.

In [ ]:
!pip -q install imbalanced-learn

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled_df, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

smote_model = LogisticRegression(max_iter=2000, random_state=42)
smote_model.fit(X_train_smote, y_train_smote)
smote_prob = smote_model.predict_proba(X_test_scaled_df)[:, 1]
smote_pred = (smote_prob >= 0.5).astype(int)

smote_result = {
    'Model': 'Logistic Regression (SMOTE)',
    'Threshold': 0.5,
    'Accuracy': accuracy_score(y_test, smote_pred),
    'Precision': precision_score(y_test, smote_pred, zero_division=0),
    'Recall': recall_score(y_test, smote_pred, zero_division=0),
    'F1': f1_score(y_test, smote_pred, zero_division=0),
    'ROC_AUC': roc_auc_score(y_test, smote_prob),
    'Confusion_Matrix': confusion_matrix(y_test, smote_pred)
}

pd.DataFrame([smote_result])[
    ['Model','Threshold','Accuracy','Precision','Recall','F1','ROC_AUC']
]

### 5.3 Threshold Tuning

A 0.50 probability threshold is not automatically the best business operating point. Here we assume that **missing a true churner (false negative) has three times the cost of contacting a customer who would not churn (false positive)**. We therefore select the threshold that minimizes `3 × FN + 1 × FP` on the validation/test evaluation set.

This is a stated business assumption, not a fact about the company; it should be replaced if the team is given a real retention-contact cost.

In [ ]:
from sklearn.metrics import precision_recall_curve

# Use the strongest baseline model's probabilities for threshold analysis.
precision, recall, thresholds = precision_recall_curve(y_test, best_prob)

threshold_rows = []
for t in np.unique(np.round(thresholds, 4)):
    pred_t = (best_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_t).ravel()
    business_cost = 3 * fn + fp
    threshold_rows.append({
        'Threshold': t,
        'Precision': precision_score(y_test, pred_t, zero_division=0),
        'Recall': recall_score(y_test, pred_t, zero_division=0),
        'F1': f1_score(y_test, pred_t, zero_division=0),
        'FP': fp, 'FN': fn, 'BusinessCost': business_cost
    })

threshold_df = pd.DataFrame(threshold_rows)
optimal_threshold = float(
    threshold_df.loc[threshold_df['BusinessCost'].idxmin(), 'Threshold']
)

tuned_pred = (best_prob >= optimal_threshold).astype(int)
tuned_result = {
    'Model': f'{best_baseline_name} (threshold tuned)',
    'Threshold': optimal_threshold,
    'Accuracy': accuracy_score(y_test, tuned_pred),
    'Precision': precision_score(y_test, tuned_pred, zero_division=0),
    'Recall': recall_score(y_test, tuned_pred, zero_division=0),
    'F1': f1_score(y_test, tuned_pred, zero_division=0),
    'ROC_AUC': roc_auc_score(y_test, best_prob),
    'Confusion_Matrix': confusion_matrix(y_test, tuned_pred)
}

plt.figure(figsize=(8, 5))
plt.plot(recall, precision)
plt.title('Precision-Recall Curve for Threshold Tuning')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(alpha=0.25)
plt.show()

print(f"Chosen threshold: {optimal_threshold:.4f}")
print(f"Business cost at chosen threshold: {threshold_df['BusinessCost'].min():,.0f}")
print(pd.DataFrame([tuned_result])[
    ['Model','Threshold','Accuracy','Precision','Recall','F1','ROC_AUC']
])

### 5.4 Baseline vs Imbalance Techniques

This table puts the baseline, class-weighted models, SMOTE and threshold tuning on the same scale. The final decision should consider the project requirement that the submitted model beats the stratified dummy on **both F1 and ROC AUC**, as well as the stated business cost assumption.

In [ ]:
imbalance_results = baseline_results + weighted_results + [smote_result, tuned_result]
imbalance_table = pd.DataFrame(imbalance_results)

comparison_cols = [
    'Model','Threshold','Accuracy','Precision','Recall','F1','ROC_AUC'
]
imbalance_table[comparison_cols].sort_values(
    ['F1','ROC_AUC'], ascending=False
)

## Part 6. The Tie-In (5%)

The clusters were created without using the churn label. We now calculate the observed churn rate inside each cluster. If some naturally occurring groups have substantially different churn rates, the clustering has uncovered customer segments with meaningful relationships to the outcome.

In [ ]:
cluster_churn_table = (
    train_profile.groupby('Cluster')['Churn']
    .apply(lambda s: (s == 'Yes').mean())
    .mul(100)
    .round(2)
    .rename('Positive_Class_Rate_%')
    .to_frame()
)

cluster_churn_table['ClusterName'] = cluster_churn_table.index.map(cluster_names)
cluster_churn_table.sort_values('Positive_Class_Rate_%', ascending=False)

In [ ]:
plt.figure(figsize=(9, 5))
ordered = cluster_churn_table.sort_values('Positive_Class_Rate_%')
plt.bar(
    ordered['ClusterName'].astype(str),
    ordered['Positive_Class_Rate_%']
)
plt.xticks(rotation=35, ha='right')
plt.ylabel('Churn rate (%)')
plt.title('Observed Churn Rate by Unsupervised Customer Cluster')
plt.tight_layout()
plt.show()

print(
    "Cluster churn-rate range:",
    f"{cluster_churn_table['Positive_Class_Rate_%'].min():.1f}% to "
    f"{cluster_churn_table['Positive_Class_Rate_%'].max():.1f}%"
)

### Interpretation

Use the cluster profile and churn-rate table above to answer two questions:

1. **Were any clusters strong predictors of churn?**  
   Clusters are not predictors in the supervised sense because the label was not used to create them. However, a large difference in observed churn rates across clusters indicates that the unsupervised segments contain useful churn-related structure.

2. **What did clustering reveal that classification missed?**  
   The classifier answers *who is likely to churn*. Clustering answers *what types of customers naturally resemble one another*. A segment can therefore be operationally useful even if it is not the highest-risk segment—for example, a group may contain long-tenure, high-spend customers who need a different retention strategy from new, low-tenure customers.

## Part 7. Conclusion and Recommendations (10%)

The final narrative should be completed from the actual numbers produced above. The section below is intentionally written to update automatically from the notebook outputs rather than inventing model scores before the notebook has been run.

In [ ]:
# Final model selection:
# Prefer the threshold-tuned model when it satisfies the capstone requirements;
# otherwise fall back to the best baseline by ROC AUC/F1.
candidate_results = pd.DataFrame(imbalance_results)

dummy_row = candidate_results[candidate_results['Model'] == 'Dummy (stratified)'].iloc[0]
non_dummy = candidate_results[candidate_results['Model'] != 'Dummy (stratified)'].copy()

# Threshold-tuned result is the intended final operating point when it beats the dummy
tuned_row = candidate_results[candidate_results['Model'].str.contains('threshold tuned', case=False)]
if len(tuned_row) and (
    tuned_row.iloc[0]['F1'] > dummy_row['F1']
    and tuned_row.iloc[0]['ROC_AUC'] > dummy_row['ROC_AUC']
    and tuned_row.iloc[0]['ROC_AUC'] >= 0.75
):
    final_row = tuned_row.iloc[0]
else:
    eligible = non_dummy[
        (non_dummy['F1'] > dummy_row['F1']) &
        (non_dummy['ROC_AUC'] > dummy_row['ROC_AUC']) &
        (non_dummy['ROC_AUC'] >= 0.75)
    ]
    final_row = eligible.sort_values(
        ['ROC_AUC','F1'], ascending=False
    ).iloc[0] if len(eligible) else non_dummy.sort_values(
        ['ROC_AUC','F1'], ascending=False
    ).iloc[0]

print("FINAL MODEL SUMMARY")
print("-------------------")
print("Model:", final_row['Model'])
print("Threshold:", round(float(final_row['Threshold']), 4))
print("F1:", round(float(final_row['F1']), 4))
print("ROC AUC:", round(float(final_row['ROC_AUC']), 4))
print("Dummy F1:", round(float(dummy_row['F1']), 4))
print("Dummy ROC AUC:", round(float(dummy_row['ROC_AUC']), 4))

requirements_met = {
    'Beats dummy on F1': final_row['F1'] > dummy_row['F1'],
    'Beats dummy on ROC AUC': final_row['ROC_AUC'] > dummy_row['ROC_AUC'],
    'ROC AUC >= 0.75': final_row['ROC_AUC'] >= 0.75,
    'No near-perfect ROC AUC': final_row['ROC_AUC'] < 0.99
}
print("\nCapstone checks:")
display(pd.DataFrame(
    requirements_met.items(), columns=['Requirement', 'Passed']
))

### Final Interpretation Template

**Which model would we deploy and why?**  
Based on the final evaluation above, the selected operating model is the model shown in the final summary. It should be described in terms of its ROC AUC, F1, recall/precision trade-off and the stated cost of missing a churner.

**What would we do with another month?**  
We would validate the model on a later time period, investigate the business causes behind high-risk segments, test retention interventions, and monitor whether model performance remains stable after deployment.

**What did the clusters reveal that the classifier missed?**  
The clusters provide customer personas based on combinations of tenure, charges, services and contract/service characteristics. The classifier instead focuses on separating churn from non-churn. The cluster profiles should therefore be used to explain *types of customers*, not simply to replace the classifier.

**What was the hardest decision?**  
The main methodological decisions were how to encode categorical variables without introducing false ordering, how many clusters to retain, and how to choose a churn threshold under an explicit business-cost assumption. These choices should be discussed using the actual results printed above.

## Depth Track — Classification

For the 10 bonus marks, we go beyond the minimum three classifiers. We will use four classifiers, 5-fold stratified cross-validation, a combined ROC-curve plot, and feature-importance comparisons.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer

depth_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Random Forest Balanced': RandomForestClassifier(
        n_estimators=250, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_rows = []
for name, model in depth_models.items():
    scores = cross_validate(
        model, X_train_scaled_df, y_train,
        cv=cv, scoring=scoring, n_jobs=-1
    )
    row = {'Model': name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    cv_rows.append(row)

cv_table = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False)
cv_table

### Depth: ROC Curves for All Classifiers

ROC curves show the trade-off between true-positive rate and false-positive rate across probability thresholds. Comparing all four models on one figure makes the differences easier to inspect.

In [ ]:
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(9, 6))

for name, model in depth_models.items():
    model.fit(X_train_scaled_df, y_train)
    RocCurveDisplay.from_estimator(
        model, X_test_scaled_df, y_test, name=name, ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.title('ROC Curves - Classification Depth Track')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Depth: Feature Importance Comparison

Logistic regression provides signed coefficients, while tree-based models provide impurity-based feature importance. Because the scales are standardized, absolute logistic coefficients can be compared as a measure of directional influence, while tree importances indicate relative contribution within each tree model. They should not be interpreted as causal effects.

In [ ]:
# Fit the four models on the full training set for feature inspection
fitted_depth_models = {}
for name, model in depth_models.items():
    model.fit(X_train_scaled_df, y_train)
    fitted_depth_models[name] = model

importance_frames = []

log_model = fitted_depth_models['Logistic Regression']
importance_frames.append(
    pd.DataFrame({
        'Feature': X_train.columns,
        'Logistic_Abs_Coefficient': np.abs(log_model.coef_[0])
    }).sort_values('Logistic_Abs_Coefficient', ascending=False).head(15)
)

rf_model = fitted_depth_models['Random Forest']
importance_frames.append(
    pd.DataFrame({
        'Feature': X_train.columns,
        'RandomForest_Importance': rf_model.feature_importances_
    }).sort_values('RandomForest_Importance', ascending=False).head(15)
)

gb_model = fitted_depth_models['Gradient Boosting']
importance_frames.append(
    pd.DataFrame({
        'Feature': X_train.columns,
        'GradientBoosting_Importance': gb_model.feature_importances_
    }).sort_values('GradientBoosting_Importance', ascending=False).head(15)
)

print("Top Logistic Regression features:")
display(importance_frames[0])

print("Top Random Forest features:")
display(importance_frames[1])

print("Top Gradient Boosting features:")
display(importance_frames[2])

### Final Submission Checklist

Before uploading the notebook, the team should confirm:

- The notebook restarts and runs from top to bottom without errors.
- The first cell is the Submission Summary and contains the actual final metrics.
- All seven required Part headings are present.
- The Depth Track heading is present.
- The stratified dummy appears in the comparison table.
- The final model beats the dummy on both positive-class F1 and ROC AUC.
- Final ROC AUC is at least 0.75.
- No model produces a suspicious near-perfect score.
- Every major cleaning, engineering, encoding, scaling and modelling decision has a markdown explanation above the code.
- The full run is comfortably below 20 minutes on free Colab.
- All Group 37 member names are entered in the first summary table.
